In [1]:
import scanpy as sc
from pathlib import Path
from matplotlib.collections import PathCollection
import gc
import numpy as np
import pandas as pd

In [7]:
PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "datasets/pseudobulk/"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
adata = sc.read_h5ad(PROJECT_ROOT / 'AIDA2.h5ad')

In [5]:
def generate_pseudo_bulk(adata):
    for cell_type in adata.obs['cell_type'].unique():
        print(cell_type)
        celltype = adata[adata.obs["cell_type"] == cell_type]
        sample_ids = celltype.obs['donor_id'].values
        sex = celltype.obs['sex'].values
        unique_samples = np.unique(sample_ids)
        
        agg = np.zeros((celltype.raw.X.shape[1], len(unique_samples)))  # Genes x Samples
        met = []
        ids = []
        n_cells = []
        smoking = []
        age = []
        ethnicity = []
        country = []
        batch = []
        
        for i, donor_id in enumerate(unique_samples):
            idx = np.where(sample_ids == donor_id)[0]
            agg[:, i] = celltype.raw.X[idx].sum(axis=0)
            met.append(np.unique(celltype.obs['sex'].values[idx])[0])
            n_cells.append(celltype.raw.X[idx].shape[0])
            ids.append(np.unique(celltype.obs['donor_id'].values[idx])[0])
            smoking.append(np.unique(celltype.obs['Smoking Status'].values[idx])[0])
            age.append(np.unique(celltype.obs['development_stage'].values[idx])[0])
            ethnicity.append(np.unique(celltype.obs['self_reported_ethnicity'].values[idx])[0])
            country.append(np.unique(celltype.obs['Country'].values[idx])[0])
            batch.append(np.unique(celltype.obs['library_id'].values[idx])[0])
        
        pseudo_bulk = pd.DataFrame(agg, index=celltype.var_names, columns=unique_samples)
        pseudo_bulk.index.name = "ENSEMBL_ID"
        metadata = pd.DataFrame({'sex': met, 'n_cells': n_cells, 'smoking_status':smoking, 'age':age,
                                 'ethnicity':ethnicity, 'country':country, 'batch':batch}, index=ids)
        metadata.index.name = "donor_id"

        pseudo_bulk.to_csv(OUTPUT_DIR / (str(cell_type) + ".csv"))
        metadata.to_csv(OUTPUT_DIR / ("coldata_" + str(cell_type) + ".csv"))
        del celltype
        gc.collect()

In [8]:
generate_pseudo_bulk(adata)

T cell
myeloid cell
natural killer cell
B cell
platelet
plasma cell
naive B cell
memory B cell
mature B cell
hematopoietic stem cell
CD14-positive monocyte
CD14-low, CD16-positive monocyte
CD1c-positive myeloid dendritic cell
CD141-positive myeloid dendritic cell
pre-conventional dendritic cell
plasmacytoid dendritic cell
CD16-positive, CD56-dim natural killer cell, human
CD8-positive, alpha-beta cytotoxic T cell
CD8-positive, alpha-beta T cell
CD16-negative, CD56-bright natural killer cell, human
CD4-positive, alpha-beta cytotoxic T cell
gamma-delta T cell
CD8-positive, alpha-beta memory T cell
CD4-positive, alpha-beta T cell
innate lymphoid cell
naive thymus-derived CD8-positive, alpha-beta T cell
mucosal invariant T cell
naive thymus-derived CD4-positive, alpha-beta T cell
central memory CD4-positive, alpha-beta T cell
effector memory CD4-positive, alpha-beta T cell
regulatory T cell
double negative T regulatory cell
